In [1]:
import os
import sys
import subprocess
import pickle
import code
import h5py

import networkx as nx
import scanpy as sc
import pandas as pd
import anndata as ad
import scipy.sparse as sp_sparse
import numpy as np
import matplotlib.pyplot as plt

import scipy.io
from scipy.io import mmread
from pkg_resources import resource_filename
from datetime import datetime
from collections import Counter
from tqdm import tqdm

# 1. 加载 autoreload 扩展
%load_ext autoreload

# 2. 设置模式为 "2" (表示自动重载所有模块)
%autoreload 2


sys.path.append('/home/liyang/BioWuYan/MethodTest/dygmamba/')

# # Data Load
data_path = "/home/liyang/BioWuYan/MethodTest/Data/All2/0process/"

print("start preprocess")

# annotation_path = "../Data/" + "annotations"

output_path = "/home/liyang/BioWuYan/MethodTest/dygmamba/result/"

os.makedirs(output_path, exist_ok=True )

start preprocess


# Data Load

In [5]:
adata_rna = ad.read_h5ad(data_path + "rna_processed.h5ad")

adata_atac = ad.read_h5ad(data_path + "atac_processed.h5ad")

pseudotime = pd.read_csv(data_path + "max_cells_lineage_Lineage1_pseudotime.csv")

pseudotime['cell_barcode'] = pseudotime['cell_barcode'].str.lower()

pseudotime.set_index("cell_barcode", inplace=True)

# Data processed

## 统一细胞

In [12]:
from src.dygmamba_preprocess import check_consistency_cell

adata_rna, adata_atac, pseudotime = check_consistency_cell(adata_rna, adata_atac, pseudotime)

In [11]:
rna_cell = {str(cell).lower().strip() for cell in adata_atac.obs_names.to_list()}
atac_cell = {str(cell).lower().strip() for cell in adata_rna.obs_names.to_list()}
pseudotime_cell = {str(cell).lower().strip() for cell in pseudotime.index.to_list()}

total_cell = list(set(rna_cell) & set(atac_cell) & set(pseudotime_cell))

adata_atac = adata_atac[total_cell,].copy()
adata_rna = adata_rna[total_cell,].copy()
pseudotime = pseudotime[pseudotime.index.str.lower().isin(total_cell)].copy()


## 选取子集

In [5]:
sub_adata_rna = adata_rna[:100,:]
sub_adata_atac = adata_atac[:100,:]
sub_cell_pseudotime = pseudotime.iloc[:100,]


# 获取调控数据

In [13]:

adata_rp_gene_peak = ad.read_h5ad(data_path + "peak_gene_rp_network.h5ad")
adata_rp_peak = ad.read_h5ad(data_path + "peak_peak_rp_network.h5ad")

In [ ]:
print(adata_rp_gene_peak)
print(adata_rp_peak)

# Filter 基因以及peak

In [ ]:
print("******************* Consistency Gene ***************************")

print(f"Before filter: adata_rna have gene: {adata_rna.shape[1]}, adata_rp_gene_preak have gene: {adata_rp_gene_peak.shape[0]}")

total_gene= set(adata_rna.var_names) & set(adata_rp_gene_peak.obs_names)
adata_rna = adata_rna[:,list(total_gene)].copy()
adata_rp_gene_peak = adata_rp_gene_peak[list(total_gene),:].copy()


print(f"After filter: adata_rna have gene: {adata_rna.shape[1]}, adata_rp_gene_preak have gene: {adata_rp_gene_peak.shape[0]}")

In [ ]:
print("******************* Consistency Peak ***************************")

print(f"Before filter: adata_atac have peak: {adata_atac.shape[1]}")
print(f"adata_rp_gene_peak have peak: {adata_rp_gene_peak.shape[0]}")
print(f"adata_rp_peak have peak: {adata_rp_peak.shape[1]}")

total_peak= set(adata_atac.var_names) & set(adata_rp_gene_peak.var_names) & set(adata_rp_peak.var_names)
adata_atac = adata_atac[:,list(total_peak)].copy()
adata_rp_gene_peak = adata_rp_gene_peak[:, list(total_peak)].copy()

adata_rp_peak = adata_rp_peak[:, list(total_peak)].copy()

adata_rp_peak = adata_rp_peak[list(total_peak), :].copy()

print(f"After filter: adata_atac have gene: {adata_atac.shape[1]}")
print(f"adata_rp_gene_peak have gene: {adata_rp_gene_peak.shape[0]}")
print(f"adata_rp_peak have peak: {adata_rp_peak.shape[1]}")

# 统计所有的节点

In [18]:
# get gene data
gene_names = adata_rna.var_names

df_genes = pd.DataFrame({
    "name" : list(gene_names),
    "type" : "gene"
})
df_genes = df_genes.sort_values(by = "name", ascending = True)

########################
# get peak data
peak_names = adata_atac.var_names
df_peaks = pd.DataFrame({
    "name" : list(peak_names),
    "type" : "peak"
})

df_peaks = df_peaks.sort_values(by = "name", ascending = True)

node_id =  pd.concat([df_genes, df_peaks], ignore_index=True)

# Data save

In [21]:
adata_atac.write_h5ad(output_path + "atac.h5ad")
adata_rna.write_h5ad(output_path + "rna.h5ad")
adata_rp_gene_peak.write_h5ad(output_path + "rp_gene_peak.h5ad")
adata_rp_peak.write_h5ad(output_path + "rp_peak.h5ad")
pseudotime.to_pickle(output_path + "cell_pseudotim.pkl")
node_id.to_pickle(output_path + "node_id.pkl")


# 综合分析

In [7]:
adata_atac = ad.read_h5ad(output_path + "atac.h5ad")

adata_rna = ad.read_h5ad(output_path + "rna.h5ad")

adata_rp_gene_peak = ad.read_h5ad(output_path + "rp_gene_peak.h5ad")

adata_rp_peak = ad.read_h5ad(output_path + "rp_peak.h5ad")

pseudotime = pd.read_pickle(output_path + "cell_pseudotim.pkl")

node_id = pd.read_pickle(output_path + "node_id.pkl")

In [8]:
with open(output_path + "node_feature_data.pkl", "rb") as f:
    load_data = pickle.load(f)

node_feature = load_data['node_feature']
# print(node_feature)

In [9]:
node_feature = load_data['node_feature']

In [2]:
import lmdb
import struct
import numpy as np
import os
import shutil
from tqdm import tqdm

In [4]:


feat_path = output_path + "edge_features.npy"
edge_label_path = output_path + "edge_labels.npy"
edge_feats = np.load(feat_path, mmap_mode="r")
edge_labels = np.load(edge_label_path, mmap_mode="r")

with open(output_path + "edge_records_data.pkl", "rb") as f:
    load_data = pickle.load(f)

edge_records = load_data['edge_records']
# print(edge_records)

# print(edge_feats)
# print(edge_labels)

graph_df = pd.read_pickle(output_path + "Graph_df.pkl")


In [7]:
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()

## test

In [ ]:
adata_rp_gene_peak

In [ ]:
from self_utils.new_regulation import Graph_data, Graph_data_optimized

Graph_df, edge_feature, edge_label, edge_records = Graph_data_optimized(adata_rna, adata_atac, pseudotime,
                        node_id, adata_rp_gene_peak, adata_rp_peak)



In [15]:
def get_targets(matrix_slice, var_names):
    if sp_sparse.issparse(matrix_slice.X):
        indices = matrix_slice.X.indices
        return var_names[indices], matrix_slice.X.data
    else:
        data = matrix_slice.X.flatten()
        mask = data != 0
        return var_names[mask], data[mask]
current_name = node_id.iloc[i, 0] # name
gene_slice = adata_rp_gene_peak[current_name, :]
target_peak_names, rp_values = get_targets(gene_slice, adata_rp_gene_peak.var_names)

In [ ]:
cell_pseudotime = pseudotime.copy()

name_to_node_idx = {name: idx + 1 for idx, name in enumerate(node_id['name'])}

time_values = cell_pseudotime.iloc[:, 0].values 

# ---------------------------------------------------------
# 3. 优化 Edge Generation (核心加速部分)
# ---------------------------------------------------------
print("Processing Edges...")

edge_records = []  # 使用 list 收集数据，最后一次性转 DataFrame
edge_features_list = [] # 收集 feature
edge_labels_list = []   # 收集 label

global_edge_idx = 0

def get_targets(matrix_slice, var_names):
    if sp_sparse.issparse(matrix_slice.X):
        indices = matrix_slice.X.indices
        return var_names[indices], matrix_slice.X.data
    else:
        data = matrix_slice.X.flatten()
        mask = data != 0
        return var_names[mask], data[mask]

for i in tqdm(range(50)):
    current_type = node_id.iloc[i, 1] # type
    current_name = node_id.iloc[i, 0] # name
    
    # ---------------- CASE 1: Gene Node ----------------
    if current_type == "gene":
        target_node_idx = i + 1
        
        try:
            # 批量获取所有连接的 peak
            gene_slice = adata_rp_gene_peak[current_name, :]
            target_peak_names, rp_values = get_targets(gene_slice, adata_rp_gene_peak.var_names)
        except KeyError:
            continue
            
        # 过滤只在 ATAC 中存在的 peak
        valid_mask = [p in adata_atac.var_names for p in target_peak_names]
        target_peak_names = target_peak_names[valid_mask]
        rp_values = rp_values[valid_mask]
        
        # 提前获取 Target Gene 的 Activity (向量)
        if current_name not in adata_rna.var_names: continue
        target_act_vec = adata_rna[:, current_name].X
        if sp_sparse.issparse(target_act_vec): target_act_vec = target_act_vec.toarray().flatten()
        else: target_act_vec = np.asarray(target_act_vec).flatten()

        # 遍历连接的 Source Peaks
        for src_peak, tmp_RP in zip(target_peak_names, rp_values):
            source_node_idx = name_to_node_idx.get(src_peak)
            if source_node_idx is None: continue
            
            # 获取 Source Peak Activity (向量)
            src_act_vec = adata_atac[:, src_peak].X
            if sp_sparse.issparse(src_act_vec): src_act_vec = src_act_vec.toarray().flatten()
            else: src_act_vec = np.asarray(src_act_vec).flatten()
            
            # *** 向量化计算逻辑 ***
            # 你的逻辑：if source_activity <= 0: continue else label=1
            # 这等价于：只保留 source > 0 的行
            mask = src_act_vec > 0
            
            if not np.any(mask): continue # 如果全为0，跳过
            
            # 应用掩码筛选数据
            valid_src_act = src_act_vec[mask]
            valid_tgt_act = target_act_vec[mask]
            valid_times = time_values[mask]
            
            # 计算 RegulationActivity
            tmp_RA = tmp_RP * valid_src_act
            
            # 批量添加到列表
            count = len(valid_times)
            for k in range(count):
                global_edge_idx += 1
                # 这里为了省内存，不建议用 dict，直接存 tuple 或 list
                edge_records.append({
                    'source_node': source_node_idx,
                    'target_node': target_node_idx,
                    'Regulation': tmp_RP,
                    'time': valid_times[k],
                    'source_activity': valid_src_act[k],
                    'target_activity': valid_tgt_act[k],
                    'RegulationActivity': tmp_RA[k],
                    'label': 1, # 你的逻辑里 else 都是 1
                    'edge_label': 0,
                    'edge_idx': global_edge_idx
                })
                
            edge_features_list.extend(tmp_RA)
            edge_labels_list.extend([1] * count)

    # ---------------- CASE 2: Peak Node ----------------
    elif current_type == "peak":
        target_node_idx = i + 1
        
        try:
            # 注意：这里你原代码写的是 adata_rp_peak，我假设这是 Peak-Peak 矩阵
            peak_slice = adata_rp_peak[current_name, :] 
            target_peak_names, rp_values = get_targets(peak_slice, adata_rp_peak.var_names)
        except KeyError:
            continue

        valid_mask = [p in adata_atac.var_names for p in target_peak_names]
        target_peak_names = target_peak_names[valid_mask]
        rp_values = rp_values[valid_mask]
        
        # 获取 Target Peak Activity (向量)
        if current_name not in adata_atac.var_names: continue
        target_act_vec = adata_atac[:, current_name].X
        if sp_sparse.issparse(target_act_vec): target_act_vec = target_act_vec.toarray().flatten()
        else: target_act_vec = np.asarray(target_act_vec).flatten()

        for src_peak, tmp_RP in zip(target_peak_names, rp_values):
            source_node_idx = name_to_node_idx.get(src_peak)
            if source_node_idx is None: continue
            
            src_act_vec = adata_atac[:, src_peak].X
            if sp_sparse.issparse(src_act_vec): src_act_vec = src_act_vec.toarray().flatten()
            else: src_act_vec = np.asarray(src_act_vec).flatten()
            
            # *** 向量化逻辑 ***
            # 你的逻辑：if src <= 0 or tgt <= 0: continue
            mask = (src_act_vec > 0) & (target_act_vec > 0)
            
            if not np.any(mask): continue
            
            valid_src_act = src_act_vec[mask]
            valid_tgt_act = target_act_vec[mask]
            valid_times = time_values[mask]
            
            tmp_RA = tmp_RP * valid_tgt_act * valid_src_act
            
            count = len(valid_times)
            # Peak-Peak 是双向边，一次加两条
            for k in range(count):
                global_edge_idx += 1
                idx1 = global_edge_idx
                global_edge_idx += 1
                idx2 = global_edge_idx
                
                # 第一条边
                edge_records.append({
                    'source_node': source_node_idx,
                    'target_node': target_node_idx,
                    'Regulation': tmp_RP,
                    'time': valid_times[k],
                    'source_activity': valid_src_act[k],
                    'target_activity': valid_tgt_act[k],
                    'RegulationActivity': tmp_RA[k],
                    'label': 1,
                    'edge_label': 1,
                    'edge_idx': idx1
                })
                # 第二条边
                edge_records.append({
                    'source_node': target_node_idx,
                    'target_node': source_node_idx,
                    'Regulation': tmp_RP,
                    'time': valid_times[k],
                    'source_activity': valid_tgt_act[k],
                    'target_activity': valid_src_act[k],
                    'RegulationActivity': tmp_RA[k],
                    'label': 1,
                    'edge_label': 1,
                    'edge_idx': idx2
                })

            edge_features_list.extend(np.repeat(tmp_RA, 2)) # 每个时间点有两条边
            edge_labels_list.extend([1] * (count * 2))


In [ ]:
edge_records

In [4]:
test_output_path = "/home/liyang/BioWuYan/MethodTest/dygmamba/test/"
feat_path = test_output_path + "edge_features.npy"
edge_label_path = test_output_path + "edge_labels.npy"


np.save(feat_path, edge_features_list)
np.save(edge_label_path, edge_labels_list)

edge_feats = np.load(feat_path, mmap_mode="r")
edge_labels = np.load(edge_label_path, mmap_mode="r")


# 构建图数据

In [ ]:
from self_utils.new_regulation import Graph_data

print("********************* strat construct graph data *******************************")

Graph_df, Node_feature, Edge_feature, Edge_label = Graph_data(adata_rna, adata_atac, pseudotime,
                    node_id, adata_rp_peak, adata_rp_gene_peak)

In [6]:
Graph_df["Unnamed"] = Graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = Graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']

In [7]:
with open(output_path + "real_temp_data.pkl", "wb") as f:
    pickle.dump({'Node_feature': Node_feature,
                    'Edge_feature': Edge_feature,
                    'New_Graph': New_Graph,
                    'Edge_label': Edge_label,
                    # 'ground_truth_set': ground_truth_set,
                    'node_id': node_id,
                    'Graph_df': Graph_df}, f)

In [9]:
with open(file_path + "real_temp_data.pkl", "rb") as f:
    load_data = pickle.load(f)

graph_df = load_data["Graph_df"]
New_Graph = load_data["New_Graph"]
Edge_label = load_data["Edge_label"]

In [ ]:
print(graph_df)
print(New_Graph)
print(Edge_label)

## 提取正边

In [12]:
index_true = np.where(Edge_label ==1)[0]

True_edge_label = Edge_label[index_true].copy()
# True_Edge_feature = Edge_feature[index_true].copy()
True_graph = New_Graph.iloc[index_true-1,:].copy()

In [ ]:
print(True_graph)

In [18]:
with open(file_path + "True_data.pkl", "wb") as f:
            pickle.dump({'Node_feature': Node_feature,
                         'Edge_feature': True_Edge_feature,
                         'New_Graph': True_graph,
                         'Edge_label': True_edge_label,
                         # 'ground_truth_set': ground_truth_set,
                         'node_id': node_id}, f)

In [ ]:
# source /home/liyang/miniconda3/etc/profile.d/conda.sh
# conda activate /home/liyang/BioWuYan/conda_env/DyGMamba/
# jupyter nbconvert --to python DygMambaPreprocess.ipynb

In [18]:
data_path = "data_result4/"

data_file = data_path + "True_data.pkl"

###########################################################
# load data
with open(data_file, "rb") as f:
    All_data2 = pickle.load(f)

Node_feature = All_data2["Node_feature"]
Edge_feature = All_data2["Edge_feature"]
Edge_feature = Edge_feature.reshape(-1,1)
New_Graph = All_data2["New_Graph"]
Edge_label = All_data2["Edge_label"]
Node_id = All_data2["node_id"]

In [ ]:
print(New_Graph)
print(Edge_label)

# Other

In [2]:

adata_atac = ad.read_h5ad(output_path + "atac.h5ad")

adata_rna = ad.read_h5ad(output_path + "rna.h5ad")

adata_rp_gene_peak = ad.read_h5ad(output_path + "rp_gene_peak.h5ad")

adata_rp_peak = ad.read_h5ad(output_path + "rp_peak.h5ad")

pseudotime = pd.read_pickle(output_path + "cell_pseudotim.pkl")

node_id = pd.read_pickle(output_path + "node_id.pkl")


In [3]:
print(adata_atac)

print(adata_rna)

print(adata_rp_gene_peak)

print(adata_rp_peak)

AnnData object with n_obs × n_vars = 446 × 71541
    var: 'Geneid', 'Chr', 'Start', 'End', 'Strand', 'Length'
AnnData object with n_obs × n_vars = 446 × 2000
    obs: 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'n_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
    uns: 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'norm'
    obsp: 'connectivities', 'distances'
AnnData object with n_obs × n_vars = 2000 × 71541
    uns: 'decay_distance', 'description', 'max_range'
AnnData object with n_obs × n_vars = 71541 × 71541
    uns: 'decay_distance', 'description', 'max_range'


In [38]:
from src.data_preprocess import analyze_score_distribution
adata_rp = ad.read_h5ad(data_path + "peak_gene_rp_network.h5ad")
adata_peak_rp = ad.read_h5ad(data_path + "peak_peak_rp_network.h5ad")

rp_stat = analyze_score_distribution(adata_rp)
rp_threshold = rp_stat["q0.75"]

binary_mask = adata_rp.X > rp_threshold
adata_rp.X = binary_mask.astype(int)

rp_peak_stat = analyze_score_distribution(adata_peak_rp)
rp_peak_threshold = rp_peak_stat["q0.75"]

peak_binary_mask = adata_peak_rp.X > rp_peak_threshold
adata_peak_rp.X = peak_binary_mask.astype(int)

adata_rp.write_h5ad(data_path + "binary_peak_gene_rp_network.h5ad")
adata_peak_rp.write_h5ad(data_path + "binary_peak_peak_rp_network.h5ad")



=== Data Score Distribution Statistics ===
  > Count (Non-zero links): 48966
  > Mean:   0.2305
  > Std:    0.2435
------------------------------
  > Min:    0.0007
  > 25%    : 0.0504
  > Median : 0.1276
  > 75%    : 0.3407
  > 90%    : 0.6223
  > 95%    : 0.7758
  > Max:    1.0000

=== Data Score Distribution Statistics ===
  > Count (Non-zero links): 1661173
  > Mean:   0.3339
  > Std:    0.2950
------------------------------
  > Min:    0.0312
  > 25%    : 0.0860
  > Median : 0.2223
  > 75%    : 0.5266
  > 90%    : 0.8346
  > 95%    : 0.9634
  > Max:    1.0000


## 筛选子集

In [6]:
adata_atac.var['n_cells'] = adata_atac.X.sum(axis=0).A1
top_peaks_idx = adata_atac.var['n_cells'].nlargest(5000).index
adata_atac = adata_atac[:, top_peaks_idx].copy()

genes_to_keep = adata_rna.var.sort_values('highly_variable_rank').head(500).index
adata_rna = adata_rna[:, genes_to_keep].copy()

pseudotime = pseudotime.iloc[list(range(0, len(pseudotime), 2)),:]


cells = pseudotime.index[list(range(0, 500, 2))]

adata_rp_gene_peak = adata_rp_gene_peak[genes_to_keep, top_peaks_idx].copy()

adata_rp_peak = adata_rp_peak[top_peaks_idx,top_peaks_idx].copy()

In [42]:
pseudotime = pseudotime.iloc[list(range(0, len(pseudotime), 2)),:]
print(pseudotime)

              pseudotime   lineage
cell_barcode                      
cell_105        0.000000  Lineage1
cell_282        0.103799  Lineage1
cell_296        0.130393  Lineage1
cell_371        0.162789  Lineage1
cell_173        0.306248  Lineage1
...                  ...       ...
cell_85        10.552336  Lineage1
cell_60        10.584224  Lineage1
cell_292       10.606858  Lineage1
cell_97        10.638032  Lineage1
cell_244       10.729193  Lineage1

[223 rows x 2 columns]


In [23]:
print(list(range(0, 500, 2)))
print(len(list(range(0, 500, 2))))

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98, 100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120, 122, 124, 126, 128, 130, 132, 134, 136, 138, 140, 142, 144, 146, 148, 150, 152, 154, 156, 158, 160, 162, 164, 166, 168, 170, 172, 174, 176, 178, 180, 182, 184, 186, 188, 190, 192, 194, 196, 198, 200, 202, 204, 206, 208, 210, 212, 214, 216, 218, 220, 222, 224, 226, 228, 230, 232, 234, 236, 238, 240, 242, 244, 246, 248, 250, 252, 254, 256, 258, 260, 262, 264, 266, 268, 270, 272, 274, 276, 278, 280, 282, 284, 286, 288, 290, 292, 294, 296, 298, 300, 302, 304, 306, 308, 310, 312, 314, 316, 318, 320, 322, 324, 326, 328, 330, 332, 334, 336, 338, 340, 342, 344, 346, 348, 350, 352, 354, 356, 358, 360, 362, 364, 366, 368, 370, 372, 374, 376, 378, 380, 382, 384, 386, 388, 390, 392, 394, 396, 398, 400, 402, 404, 406, 408, 410, 412, 414, 416, 418, 420,